In [ ]:
# Thư viện cho Selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service 
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException # <<< THÊM IMPORT NÀY (nếu bạn dùng WebDriverWait ở cell nào đó)

# Thư viện cho parsing HTML
from bs4 import BeautifulSoup

# Thư viện tiện ích khác
import time
import json
from datetime import datetime, timedelta 
import re
import pandas as pd
import numpy as np

In [23]:
TARGET_URL = "https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-ho-chi-minh-l2?exp=1&type_keyword=0&sba=1&locations=l2"
# Đặt tên file output dựa trên thời gian chạy để tránh ghi đè
TIMESTAMP_STR = datetime.now().strftime('%Y%m%d_%H%M%S')
HTML_OUTPUT_FILENAME = f"topcv_selenium_page_{TIMESTAMP_STR}.html"
JSON_OUTPUT_FILENAME = f"topcv_jobs_data_{TIMESTAMP_STR}.json" 

print(f"URL Mục tiêu: {TARGET_URL}")
print(f"File HTML sẽ được lưu là: {HTML_OUTPUT_FILENAME}")
print(f"File JSON sẽ được lưu là: {JSON_OUTPUT_FILENAME}") 

URL Mục tiêu: https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-ho-chi-minh-l2?exp=1&type_keyword=0&sba=1&locations=l2
File HTML sẽ được lưu là: topcv_selenium_page_20250521_213822.html
File JSON sẽ được lưu là: topcv_jobs_data_20250521_213822.json


# Cell 2: Configure and Initialize Selenium WebDriver


In [24]:

print("Đang cấu hình Chrome Options ........")
chrome_options = Options()

# --- Các tùy chọn quan trọng ---
# 1. Chạy ở chế độ "headless" (không mở cửa sổ trình duyệt vật lý)
#    Khi debug, bạn nên comment dòng này lại để có thể thấy trình duyệt hoạt động.
# chrome_options.add_argument("--headless") 

# 2. Các cờ thường cần thiết khi chạy trong môi trường Linux hoặc container (như Docker)
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage") # Khắc phục lỗi liên quan đến /dev/shm khi thiếu tài nguyên

# 3. Tắt GPU, thường không cần thiết cho scraping và có thể cải thiện hiệu suất/ổn định
chrome_options.add_argument("--disable-gpu")

# 4. Đặt User-Agent giống như trình duyệt thông thường để tránh bị phát hiện là bot
#    Bạn có thể thay bằng User-Agent từ trình duyệt của mình nếu muốn
chrome_options.add_argument("user-agent=Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36")

# 5. Đặt kích thước cửa sổ (có thể quan trọng đối với một số trang web responsive)
chrome_options.add_argument("--window-size=1920,1080")

# 6. (Tùy chọn) Chạy ở chế độ ẩn danh
# chrome_options.add_argument("--incognito")

# 7. (Tùy chọn) Tắt thông báo "Chrome is being controlled by automated test software"
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)

# 8. (Tùy chọn) Tắt hình ảnh để tải trang nhanh hơn (nếu không cần hình ảnh)
# chrome_options.add_argument('--blink-settings=imagesEnabled=false')

print("Chrome Options đã được cấu hình.")

# --- Khởi tạo WebDriver ---
# Selenium 4 sẽ tự động tìm chromedriver nếu nó nằm trong PATH hệ thống.
# Nếu bạn đặt chromedriver ở một vị trí cụ thể và không có trong PATH, bạn cần dùng Service:
# service = Service(executable_path='/duong/dan/toi/chromedriver')
# driver = webdriver.Chrome(service=service, options=chrome_options)

try:
    print("Đang khởi tạo WebDriver...")
    driver = webdriver.Chrome(options=chrome_options)
    print("WebDriver đã được khởi tạo thành công!")
except Exception as e:
    print(f"Lỗi khi khởi tạo WebDriver: {e}")
    print("Hãy đảm bảo ChromeDriver đã được cài đặt đúng và phiên bản tương thích với Chrome của bạn.")
    print("Nếu ChromeDriver không nằm trong PATH, bạn cần chỉ định 'executable_path' thông qua 'Service'.")
    driver = None # Đặt driver là None nếu khởi tạo lỗi

# Để đảm bảo driver được đóng nếu notebook bị ngắt đột ngột hoặc cell này chạy lại nhiều lần
# chúng ta sẽ lưu trữ driver vào một biến global (hơi xấu nhưng tiện cho notebook)
# hoặc tốt hơn là quản lý nó cẩn thận trong các cell sau.
# Hiện tại, chúng ta sẽ đóng nó ở cell cuối cùng.

Đang cấu hình Chrome Options ........
Chrome Options đã được cấu hình.
Đang khởi tạo WebDriver...
WebDriver đã được khởi tạo thành công!


# Cell 3: Navigate to Target URL, Wait, and Get Page Source


In [25]:
#Kiểm tra xem biến drivẻ từ cell trước có tồn tại và không phải là None không

if 'driver' in locals() and driver is not None:
    try:
        print(f"Đang điều khiển trình duyệt truy cập URL: {TARGET_URL}")
        driver.get(TARGET_URL) # Lệnh để trình duyệt mở URL
        print(f"Đã gửi yêu cầu tải trang. URL trên trình duyệt tên là: {driver.current_url}")
        
        # Chờ một khoảng thời gian cố định để trang tải hoàn chỉnh (bao gồm JavaScript)
        # Đối với các trang phức tạp, ta có thể cần phải tăng thời gian này
        # hoặc sử dụng WebDriverWait để đợi một element cụ thể xuất hiện.
        wait_time_seconds = 14 # Thử với 14 giây, có thể tăng nếu cần
        print(f"Đang đợi {wait_time_seconds} giây để nội dung trang tải hoàn chỉnh...")
        time.sleep(wait_time_seconds)
        print("Thời gian chờ đã kết thúc.")
        
        # Lấy mã nguồn HTML của trang sau khi đã tải và JavaScript (nếu có) đã chạy
        print("Đang lấy mã nguồn HTML của trang (page_source)...")
        page_source_html = driver.page_source
        
        if page_source_html:
            print(f"Đã lấy được page_source HTML (kích thước: {len(page_source_html)} bytes).")
            
            # Lưu nội dung HTML ra file để kiểm tra và tìm selector sau này
            with open(HTML_OUTPUT_FILENAME, "w", encoding="utf-8") as f:
                f.write(page_source_html)
            print(f"Nội dung HTML của trang TopCV đã được lưu vào file: '{HTML_OUTPUT_FILENAME}'")
            print("Hãy mở file này bằng trình duyệt để kiểm tra nội dung và chuẩn bị cho việc tìm selector.")
        else:
            print("Không lấy được page_source HTML. Trang có thể chưa tải xong hoặc có lỗi.")
    except Exception as e:
        print(f"Lỗi trong quá trình truy cập URL hoặc lấy page_source: {e}")
        # Cân nhắc đóng driver nếu có lỗi nghiêm trọng ở đây, 
        # nhưng chúng ta sẽ đóng ở cell cuối cùng để đảm bảo.
else:
    print("Biến 'driver' không tồn tại hoặc chưa được khởi tạo. Hãy chạy lại Cell 2.")

Đang điều khiển trình duyệt truy cập URL: https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-ho-chi-minh-l2?exp=1&type_keyword=0&sba=1&locations=l2
Đã gửi yêu cầu tải trang. URL trên trình duyệt tên là: https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-ho-chi-minh-l2?exp=1&type_keyword=0&sba=1&locations=l2
Đang đợi 14 giây để nội dung trang tải hoàn chỉnh...
Thời gian chờ đã kết thúc.
Đang lấy mã nguồn HTML của trang (page_source)...
Đã lấy được page_source HTML (kích thước: 1454110 bytes).
Nội dung HTML của trang TopCV đã được lưu vào file: 'topcv_selenium_page_20250521_213822.html'
Hãy mở file này bằng trình duyệt để kiểm tra nội dung và chuẩn bị cho việc tìm selector.


# Cell 4: Parse HTML with BeautifulSoup and Extract Data
cell này rất quan trọng, cần đọc hiểu rõ, đặc biệt là ở phần Location

In [26]:
if 'page_source_html' in locals() and page_source_html is not None:
    print(f"Đang thực hiện phân tích (Parse) HTML với (kích thước là: {len(page_source_html)} bytes) bằng BeautifulSoup...")
    soup = BeautifulSoup(page_source_html, 'html.parser')
    
    all_jobs_data = [] 
    
    # Lấy ngày scrape hiện tại một lần cho tất cả các job trong lần chạy này
    current_scrape_date = datetime.now().strftime('%Y-%m-%d') # Format YYYY-MM-DD
    # thêm cả giờ: 
    current_scrape_datetime = datetime.now().isoformat()
    
    job_list_container = soup.find('div', class_='box-job-list')
        
    if job_list_container:
        print(f"Đã tìm thấy container 'div.box-job-list'.")
        job_item_elements = job_list_container.find_all('div', class_='job-item-search-result')
        print(f"Tìm thấy tổng cộng {len(job_item_elements)} job items trong 'div.box-job-list'.")
        
        if not job_item_elements:
             print("!!! Cảnh báo: Tìm thấy 'div.box-job-list' nhưng không có 'div.job-item-search-result' nào bên trong.")
             
        # ĐỊNH NGHĨA DANH SÁCH QUẬN HUYỆN TP.HCM
        hcm_districts = [
            "Quận 1", "Quận 2", "Quận 3", "Quận 4", "Quận 5", "Quận 6", "Quận 7", "Quận 8", 
            "Quận 9", "Quận 10", "Quận 11", "Quận 12",
            "Thủ Đức", "Bình Thạnh", "Tân Bình", "Tân Phú", "Phú Nhuận", "Gò Vấp", "Bình Tân",
            "Hóc Môn", "Củ Chi", "Nhà Bè", "Bình Chánh", "Cần Giờ", "Thành phố Thủ Đức" # Thêm "Thành phố Thủ Đức"
        ]
        # Chuẩn hóa để khớp không phân biệt chữ hoa/thường và tiền tố "Quận", "TP."
        hcm_districts_normalized_for_match = [d.lower().replace("quận ", "").replace("tp. ", "").replace("thành phố ", "") for d in hcm_districts]

        for index, item_element in enumerate(job_item_elements):
            print(f"\n --- Đang xử lý job item thứ {index + 1}/{len(job_item_elements)} ---")
            job_data = {
                'job_title': None, 'job_url': None, 'company_name': None,
                'salary': None, 
                'location_raw': None, # Lưu trữ location gốc từ span.city-text
                'city_main': None,    # Tỉnh/Thành phố chính
                'district': None,     # Quận/Huyện
                'experience': None, 'post_date': None,
                'scrape_date': current_scrape_date, # Ngày scrape
            }
            
            # --- Tên công việc (Job Title) và URL chi tiết ---
            # Giả sử title_block là một div có class 'title-block' (hoặc tương tự) chứa h3.title
            title_block_div = item_element.find('div', class_='title-block') # Kiểm tra lại class này
            if title_block_div:
                title_h3 = title_block_div.find('h3', class_=['title', 'title highlight']) # Có thể có hoặc không có 'highlight'
                if title_h3:
                    title_tag_a = title_h3.find('a')
                    if title_tag_a:
                        # Job title có thể nằm trong span bên trong a, hoặc text trực tiếp của a
                        job_title_span = title_tag_a.find('span') 
                        if job_title_span:
                            job_data['job_title'] = job_title_span.get_text(strip=True)
                        else: 
                            job_data['job_title'] = title_tag_a.get_text(strip=True) 
                        
                        job_data['job_url'] = title_tag_a.get('href')
                        if job_data['job_url'] and job_data['job_url'].startswith('/'):
                            job_data['job_url'] = "https://www.topcv.vn" + job_data['job_url']
            
            if job_data['job_title']: print(f"  Tên công việc (Job title) là: {job_data['job_title']}")
            else: print("  Không tìm thấy tên công việc (Job title).")
            if job_data['job_url']: print(f"  URL chi tiết công việc là: {job_data['job_url']}")
            # else: print("  Không tìm thấy URL chi tiết công việc.") # Không cần thiết nếu title không có

            # --- Tên công ty (Company Name) ---
            # Thử tìm thẻ a chứa tên công ty trực tiếp từ item_element
            company_tag_a = item_element.find('a', class_=['company', 'company job-pro'])
            if company_tag_a:
                company_name_span = company_tag_a.find('span', class_='company-name')
                if company_name_span:
                    job_data['company_name'] = company_name_span.get_text(strip=True)
                        
            if job_data['company_name']: print(f"  Tên công ty (Company name) là: {job_data['company_name']}")
            else: print("  Không tìm thấy tên công ty (Company name) nào.")    
            
            # === Mức lương (salary) ===
            salary_tag = item_element.find(['span', 'div', 'p', 'label'], class_='title-salary')
            if salary_tag:
                job_data['salary'] = salary_tag.get_text(strip=True).replace('\n', '').replace('\r', '').strip()
            
            if job_data['salary']: print(f"  Mức lương (Salary) là: {job_data['salary']}")
            else: print("  Không tìm thấy mức lương (Salary).")
            
            # === Địa chỉ (location) - SỬA ĐỔI VÀ PHÁT TRIỂN THÊM LOGIC ===
            location_label_tag = item_element.find('label', class_='address truncate')
            raw_location_from_span = None
            
            if location_label_tag:
                # Lấy text gốc từ span.city-text để lưu lại nếu cần
                location_span_tag = location_label_tag.find('span', class_='city-text')
                if location_span_tag:
                    raw_location_from_span = location_span_tag.get_text(strip=True)
                    job_data['location_raw'] = raw_location_from_span # Lưu lại text gốc
                
                # Ưu tiên lấy thông tin chi tiết từ data-original-title
                tooltip_title_attr = location_label_tag.get('data-original-title')
                if tooltip_title_attr:
                    tooltip_soup = BeautifulSoup(tooltip_title_attr, 'html.parser')
                    p_tag_tooltip = tooltip_soup.find('p')
                    if p_tag_tooltip:
                        # Xử lý trường hợp có nhiều địa điểm cách nhau bởi <br/> trong tooltip
                        locations_in_tooltip = []
                        current_location_segment = ""
                        for content_part in p_tag_tooltip.contents:
                            if isinstance(content_part, str):
                                current_location_segment += content_part.strip()
                            elif content_part.name == 'br':
                                if current_location_segment:
                                    locations_in_tooltip.append(current_location_segment.strip())
                                current_location_segment = "" # Reset for next segment
                        if current_location_segment: # Add the last segment
                            locations_in_tooltip.append(current_location_segment.strip())

                        # print(f"DEBUG: Locations from tooltip: {locations_in_tooltip}")

                        is_hcm_primary = False
                        districts_found_in_hcm = []

                        for loc_detail_str in locations_in_tooltip:
                            loc_detail_str_lower = loc_detail_str.lower()
                            if "hồ chí minh" in loc_detail_str_lower or "hcm" in loc_detail_str_lower:
                                is_hcm_primary = True
                                # Trích xuất quận từ "Hồ Chí Minh: Tên Quận"
                                district_match = re.search(r'(?:hồ chí minh|hcm)\s*:\s*(.+)', loc_detail_str, re.IGNORECASE)
                                if district_match:
                                    district_name_raw = district_match.group(1).strip().lower()
                                    for i_dist, dist_norm_val in enumerate(hcm_districts_normalized_for_match):
                                        if re.search(r'\b' + re.escape(dist_norm_val) + r'\b', district_name_raw):
                                            districts_found_in_hcm.append(hcm_districts[i_dist])
                                            break 
                        
                        if is_hcm_primary:
                            job_data['city_main'] = "Hồ Chí Minh"
                            if districts_found_in_hcm:
                                job_data['district'] = ", ".join(sorted(list(set(districts_found_in_hcm)))) # Lưu các quận tìm được, nối bằng dấu phẩy nếu nhiều
                            # else: job_data['district'] vẫn là None nếu chỉ có "Hồ Chí Minh" chung chung
                        # (Có thể thêm logic xử lý nếu địa điểm chính không phải HCM mà bạn vẫn muốn ghi nhận)
                
                # Fallback: Nếu không lấy được quận từ tooltip HOẶC không có tooltip, 
                # và city_main là HCM, thử lấy quận từ text của span.city-text (nếu nó là tên quận)
                if job_data['city_main'] == "Hồ Chí Minh" and not job_data['district'] and raw_location_from_span:
                    raw_location_from_span_lower = raw_location_from_span.lower()
                    for i_dist_span, dist_norm_val_span in enumerate(hcm_districts_normalized_for_match):
                        if re.search(r'\b' + re.escape(dist_norm_val_span) + r'\b', raw_location_from_span_lower):
                            # Chỉ gán nếu text trong span là tên quận và không phải "hồ chí minh"
                            if raw_location_from_span_lower not in ["hồ chí minh", "hcm"]:
                                job_data['district'] = hcm_districts[i_dist_span]
                                break
                elif pd.isna(job_data['city_main']) and raw_location_from_span: # Nếu city_main chưa được set và có text từ span
                    if "hồ chí minh" in raw_location_from_span.lower() or "hcm" in raw_location_from_span.lower():
                        job_data['city_main'] = "Hồ Chí Minh"

            # In kết quả location
            if job_data['city_main']: print(f"  Thành phố là: {job_data['city_main']}")
            else: print("Không xác định được thành phố.")
            if job_data['district']: print(f"  Quận/Huyện là: {job_data['district']}")
            else: print("Không tìm thấy thông tin quận/huyện cụ thể cho TP.HCM.")
            # if job_data['location_raw']: print(f"  Location (raw từ span): {job_data['location_raw']}") # Để debug

            # === Kinh nghiệm (experience) ===
            exp_label = item_element.find('label', class_='exp')
            if exp_label:
                exp_span = exp_label.find('span')
                if exp_span:
                    job_data['experience'] = exp_span.get_text(strip=True)
            
            if job_data['experience']: print(f"  Kinh nghiệm (Experience) là: {job_data['experience']}")
            else: print("Không tìm thấy kinh nghiệm (Experience).")
                
            # === Ngày đăng (post_date) ===
            date_tag = item_element.find('label', class_='address mobile-hidden label-update')
            if date_tag:
                raw_date_text = date_tag.get_text(strip=True)
                job_data['post_date'] = raw_date_text.replace('Đăng', '').strip() 
            
            if job_data['post_date']: print(f"  Ngày đăng (Post date) là: {job_data['post_date']}")
            else: print("  Không tìm thấy ngày đăng (Post date).") 
            
            # <<< IN RA NGÀY SCRAPE CHO TỪNG JOB ĐỂ KIỂM TRA >>>
            print(f"  Ngày Scrape: {job_data['scrape_date']}")
                
            if job_data['job_title'] and job_data['job_url']:
                all_jobs_data.append(job_data)
                print(f"  Đã thêm job item thứ {index + 1} vào danh sách.")
            else:
                print(f"  !! Job item thứ {index + 1} thiếu thông tin Title hoặc URL, không được thêm vào kết quả.")
                
        print(f'\n Đã xử lý xong {len(job_item_elements)} job items.')
        if all_jobs_data:
            print(f"Tổng số jobs hợp lệ đã được trích xuất là: {len(all_jobs_data)}")
            print("\n Ví dụ 3 jobs đầu tiên đã trích xuất: ")
            for i, job_sample in enumerate(all_jobs_data[:3]):
                print(f"  --- Job {i + 1} ---")
                for key, value in job_sample.items(): 
                    print(f"    {key.replace('_', ' ').title()}: {value}")
        else:
            print("Không có job nào được trích xuất hợp lệ.")
            
    else:
        print("Không tìm thấy container 'div.box-job-list'. Vui lòng kiểm tra lại file HTML và selector.")

else:
    print("Biến 'page_source_html' không tồn tại hoặc rỗng. Hãy chạy lại Cell 3.")

Đang thực hiện phân tích (Parse) HTML với (kích thước là: 1454110 bytes) bằng BeautifulSoup...
Đã tìm thấy container 'div.box-job-list'.
Tìm thấy tổng cộng 50 job items trong 'div.box-job-list'.

 --- Đang xử lý job item thứ 1/50 ---
  Tên công việc (Job title) là: Nhân Viên Kinh Doanh/ Sale Bất Động Sản, Lương Cứng 5Tr- 10Tr, Hoa Hồng 2,5%/ Giao Dịch, Thu Nhập Hấ...
  URL chi tiết công việc là: https://www.topcv.vn/viec-lam/nhan-vien-kinh-doanh-sale-bat-dong-san-luong-cung-5tr-10tr-hoa-hong-2-5-giao-dich-thu-nhap-hap-dan-lam-viec-tai-hcm/1732398.html?ta_source=JobSearchList_LinkDetail&u_sr_id=8EWgUyBXOvRwFuukQ1QMFfEopkeZgJlHEwrZR8Lb_1747838303
  Tên công ty (Company name) là: CÔNG TY CỔ PHẦN KINH DOANH BẤT ĐỘNG SẢN LUNA HOLDINGS
  Mức lương (Salary) là: 40 - 50 triệu
  Thành phố là: Hồ Chí Minh
  Quận/Huyện là: Bình Tân
  Kinh nghiệm (Experience) là: Không yêu cầu
  Ngày đăng (Post date) là: 5 ngày trước
  Ngày Scrape: 2025-05-21
  Đã thêm job item thứ 1 vào danh sách.

 --- Đang xử l

# Cell 5: Save Extracted Data to JSON File and Close Webdriver
Mục đích: Lưu toàn bộ dữ liệu vào file JSON cuối cùng và quan trọng là đóng trình duyệt Selenium

In [27]:
# Kiểm tra xem biến all_jobs_data từ Cell 4 có tồn tại và có dữ liệu không
if 'all_jobs_data' in locals() and all_jobs_data:
    try:
        # Sử dụng JSON_OUTPUT_FILENAME đã định nghĩa ở Cell 1
        print(f"\nĐang chuẩn bị lưu {len(all_jobs_data)} job items (từ trang 1) vào file JSON...")
        with open(JSON_OUTPUT_FILENAME, 'w', encoding='utf-8') as f_json:
            json.dump(all_jobs_data, f_json, ensure_ascii=False, indent=4)
        print(f"Thành công! Dữ liệu trang 1 đã được lưu vào file: '{JSON_OUTPUT_FILENAME}'")
        
        print("\nVí dụ 3 job đầu tiên đã trích xuất (từ trang 1):")
        for i, job_sample in enumerate(all_jobs_data[:3]):
            print(f"  --- Job {i+1} ---")
            for key, value in job_sample.items():
                print(f"    {key.replace('_', ' ').title()}: {value}")
                    
    except Exception as e_save:
        print(f"Lỗi khi lưu file JSON: {e_save}")
elif 'all_jobs_data' in locals() and not all_jobs_data:
     print("Không có dữ liệu job nào để lưu (danh sách all_jobs_data từ trang 1 rỗng).")
else:
    print("Biến 'all_jobs_data' không tồn tại. Hãy đảm bảo Cell 4 (cũ) đã chạy đúng.")

# Đóng WebDriver (Rất quan trọng!)
if 'driver' in locals() and driver is not None:
    try:
        print("\nĐang đóng WebDriver...")
        driver.quit()
        print("WebDriver đã được đóng thành công.")
        # Đặt lại biến driver thành None để tránh sử dụng lại ở cell khác nếu chạy lại
        # và cũng để cell này có thể chạy lại mà không báo lỗi driver đã đóng
        driver = None  
    except Exception as e_quit:
        print(f"Lỗi khi đóng WebDriver: {e_quit}")
else:
    print("\nWebDriver không tồn tại hoặc đã được đóng trước đó.")

print("\n--- HOÀN TẤT QUÁ TRÌNH SCRAPING TRANG 1 ---")


Đang chuẩn bị lưu 50 job items (từ trang 1) vào file JSON...
Thành công! Dữ liệu trang 1 đã được lưu vào file: 'topcv_jobs_data_20250521_213822.json'

Ví dụ 3 job đầu tiên đã trích xuất (từ trang 1):
  --- Job 1 ---
    Job Title: Nhân Viên Kinh Doanh/ Sale Bất Động Sản, Lương Cứng 5Tr- 10Tr, Hoa Hồng 2,5%/ Giao Dịch, Thu Nhập Hấ...
    Job Url: https://www.topcv.vn/viec-lam/nhan-vien-kinh-doanh-sale-bat-dong-san-luong-cung-5tr-10tr-hoa-hong-2-5-giao-dich-thu-nhap-hap-dan-lam-viec-tai-hcm/1732398.html?ta_source=JobSearchList_LinkDetail&u_sr_id=8EWgUyBXOvRwFuukQ1QMFfEopkeZgJlHEwrZR8Lb_1747838303
    Company Name: CÔNG TY CỔ PHẦN KINH DOANH BẤT ĐỘNG SẢN LUNA HOLDINGS
    Salary: 40 - 50 triệu
    Location Raw: Hồ Chí Minh
    City Main: Hồ Chí Minh
    District: Bình Tân
    Experience: Không yêu cầu
    Post Date: 5 ngày trước
    Scrape Date: 2025-05-21
  --- Job 2 ---
    Job Title: Nhân Viên Kinh Doanh/ Sales Bất Động Sản Nhà Phố - Thu Nhập Hấp Dẫn Từ 50M+++
    Job Url: https://www

# Cell 6: Load Data from JSON and Initial Processing with Pandas

In [28]:
if 'JSON_OUTPUT_FILENAME' not in locals() or not JSON_OUTPUT_FILENAME:
    print("Biến JSON_OUTPUT_FILENAME không tồn tại. Hãy chạy lại Cell 1 để định nghĩa biến này.")
    raise NameError("Biến JSON_OUTPUT_FILENAME không tồn tại.")

print(f"Đang đọc dữ liệu từ file JSON: {JSON_OUTPUT_FILENAME}...")

try:
    # Đọc dữ liệu từ file JSON vào Pandas DataFrame
    df_jobs = pd.read_json(JSON_OUTPUT_FILENAME, encoding='utf-8') 
    # Kiểm tra xem dữ liệu đã được đọc thành công chưa
    if df_jobs.empty:
        print("!!! Cảnh báo: Không có dữ liệu nào được đọc từ file JSON. Vui lòng kiểm tra lại file.")
    
    print(f"Đã đọc thành công {len(df_jobs)} job items từ file JSON.")
    
    # === hiển thị thông tin cơ bản về DataFrame ===
    print("\nThông tin cơ bản về DataFrame (df_jobs.info()): ")
    df_jobs.info()
    
    print('\n ===5 dòng dữ liệu đầu tiên (df_jobs.head()) ===')
    display(df_jobs.head(5))
    
    # Kiểm tra các giá trị bị thiếu (NaN) cho mỗi cột
    print("\n--- Kiểm tra các giá trị bị thiếu (NaN) cho mỗi cột ---")
    print(df_jobs.isnull().sum())
    
    # Bắt đầu các bứớc làm sạch và chuẩn hóa dữ liệu cơ bản
    print("\n Bắt đầu quá trình làm sạch và chuẩn hóa dữ liệu...")
    
    # 1. Xử lý cột 'salary
    # Ví dụ: "10 - 20 triệu", "Thoả thuận", "Tới 100 triệu"
    # Mục tiêu có thể là tách ra mức lương min, max, đơn vị, hoặc giữ nguyên nếu là "Thoả thuận"
    # Đây là một bước phức tạp và sẽ cần nhiều logic hơn, tạm thời chúng ta chỉ xem xét
    print("\nXem xét cột 'salary':")
    print(df_jobs['salary'].value_counts(dropna=False).head(10)) # Xem 10 giá trị phổ biến nhất (bao gồm cả NaN)
    
    # 2. Xử lý cột 'location' (Địa điểm)
    print("\nXem xét cột 'location_raw' (dữ liệu gốc từ span):")
    print(df_jobs['location_raw'].value_counts(dropna=False).head(10))
    
    print("\nXem xét cột 'city_main':")
    print(df_jobs['city_main'].value_counts(dropna=False).head(10))

    print("\nXem xét cột 'district' (cho TP.HCM):")
    # Chỉ in value_counts của district nếu cột tồn tại và có ít nhất 1 job ở HCM
    if 'city_main' in df_jobs.columns and "Hồ Chí Minh" in df_jobs['city_main'].unique():
        print(df_jobs[df_jobs['city_main'] == "Hồ Chí Minh"]['district'].value_counts(dropna=False).head(15)) # In nhiều hơn để xem các quận
    else:
        print("Không có cột 'city_main' hoặc không có job nào ở Hồ Chí Minh để hiển thị quận.")

    # 3. Xử lý cột 'experience' (Kinh nghiệm)
    # Ví dụ: "Không yêu cầu", "1 năm", "Dưới 1 năm"
    # Có thể cần chuẩn hóa (ví dụ: "Dưới 1 năm" -> "<1 year")
    print("\nXem xét cột 'experience':")
    print(df_jobs['experience'].value_counts(dropna=False).head(10))
    
    # 4. Xử lý cột 'post_date' (Ngày đăng)
    # Ví dụ: "1 ngày trước", "1 tuần trước", "Hôm qua"
    # Mục tiêu có thể là chuyển đổi thành một ngày cụ thể (datetime object)
    print("\nXem xét cột 'post_date':")
    print(df_jobs['post_date'].value_counts(dropna=False).head(10))
    
    
    # --- Gợi ý các bước xử lý tiếp theo (sẽ làm ở các cell sau) ---
    # - Chuyển đổi kiểu dữ liệu (ví dụ: salary_min, salary_max sang số)
    # - Trích xuất keywords từ 'job_title' (nếu cần)
    # - Tạo các cột mới (ví dụ: tỉnh thành chính từ 'location')
    # - Xử lý các giá trị NaN (điền giá trị mặc định, hoặc loại bỏ hàng/cột nếu phù hợp)
    
    print("\n--- Kết thúc xem xét dữ liệu ban đầu ---")
    print("DataFrame 'df_jobs' đã sẵn sàng cho các bước xử lý chi tiết hơn.")
    
except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file '{JSON_OUTPUT_FILENAME}'. Hãy đảm bảo file đã được lưu đúng cách từ cell trước.")
except KeyError as e:
    print(f"Lỗi KeyError: Cột {e} không tồn tại trong DataFrame. Có thể tên cột trong file JSON khác với dự kiến.")
    print("Kiểm tra lại cấu trúc file JSON hoặc tên cột trong code.")
except Exception as e:
    print(f"Lỗi khi đọc hoặc xử lý file JSON: {e}")

Đang đọc dữ liệu từ file JSON: topcv_jobs_data_20250521_213822.json...
Đã đọc thành công 50 job items từ file JSON.

Thông tin cơ bản về DataFrame (df_jobs.info()): 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   job_title     50 non-null     object
 1   job_url       50 non-null     object
 2   company_name  50 non-null     object
 3   salary        50 non-null     object
 4   location_raw  50 non-null     object
 5   city_main     50 non-null     object
 6   district      48 non-null     object
 7   experience    50 non-null     object
 8   post_date     50 non-null     object
 9   scrape_date   50 non-null     object
dtypes: object(10)
memory usage: 4.0+ KB

 ===5 dòng dữ liệu đầu tiên (df_jobs.head()) ===


,job_title,job_url,company_name,salary,location_raw,city_main,district,experience,post_date,scrape_date
0,"Nhân Viên Kinh Doanh/ Sale Bất Động Sản, Lương...",https://www.topcv.vn/viec-lam/nhan-vien-kinh-d...,CÔNG TY CỔ PHẦN KINH DOANH BẤT ĐỘNG SẢN LUNA H...,40 - 50 triệu,Hồ Chí Minh,Hồ Chí Minh,Bình Tân,Không yêu cầu,5 ngày trước,2025-05-21
1,Nhân Viên Kinh Doanh/ Sales Bất Động Sản Nhà P...,https://www.topcv.vn/viec-lam/nhan-vien-kinh-d...,CÔNG TY CỔ PHẦN BẤT ĐỘNG SẢN TUẤN 123 MIỀN NAM,50 - 200 triệu,"Hồ Chí Minh, Bình Dương",Hồ Chí Minh,Quận 7,Không yêu cầu,6 ngày trước,2025-05-21
2,Nhân Viên Kinh Doanh/ Tư Vấn Bất Động Sản - Th...,https://www.topcv.vn/brand/bdsanewhome/tuyen-d...,Công ty Cổ phần Bất động sản A New Home,Tới 80 triệu,Hồ Chí Minh,Hồ Chí Minh,Thủ Đức,Không yêu cầu,1 tuần trước,2025-05-21
3,Nhân Viên Kinh Doanh / Sales / Tư Vấn Bất Độn...,https://www.topcv.vn/viec-lam/nhan-vien-kinh-d...,CÔNG TY CỔ PHẦN ĐẦU TƯ VÀ DỊCH VỤ SAIGON REAL,Từ 50 triệu,Hồ Chí Minh,Hồ Chí Minh,Quận 2,Không yêu cầu,1 tuần trước,2025-05-21
4,Nhân Viên Kinh Doanh Bất Động Sản / Môi Giới B...,https://www.topcv.vn/viec-lam/nhan-vien-kinh-d...,CÔNG TY CỔ PHẦN DAPDANCE,Thoả thuận,Hồ Chí Minh,Hồ Chí Minh,Quận 2,Không yêu cầu,1 tuần trước,2025-05-21



--- Kiểm tra các giá trị bị thiếu (NaN) cho mỗi cột ---
job_title       0
job_url         0
company_name    0
salary          0
location_raw    0
city_main       0
district        2
experience      0
post_date       0
scrape_date     0
dtype: int64

 Bắt đầu quá trình làm sạch và chuẩn hóa dữ liệu...

Xem xét cột 'salary':
salary
Thoả thuận        6
10 - 30 triệu     3
40 - 50 triệu     2
10 - 13 triệu     2
50 - 200 triệu    2
7 - 10 triệu      2
Từ 15 triệu       2
Tới 100 triệu     2
Tới 80 triệu      1
Từ 50 triệu       1
Name: count, dtype: int64

Xem xét cột 'location_raw' (dữ liệu gốc từ span):
location_raw
Hồ Chí Minh                 41
Hồ Chí Minh, Hà Nội          5
Hồ Chí Minh & 2 nơi khác     2
Hồ Chí Minh, Bình Dương      1
Hồ Chí Minh & 3 nơi khác     1
Name: count, dtype: int64

Xem xét cột 'city_main':
city_main
Hồ Chí Minh    50
Name: count, dtype: int64

Xem xét cột 'district' (cho TP.HCM):
district
Thủ Đức       11
Quận 3         5
Bình Tân       4
Quận 7         4
Q

# Cell 7: Detailed Processing for 'salary' column

In [29]:
# Import numpy để sử dụng np.nan cho giá trị thiếu (tùy chọn, None cũng được)

if 'df_jobs' in locals() and not df_jobs.empty:
    print("--- Bắt đầu xử lý cột 'salary' ---")
    
    # Tạo các cột mới cho lương min, max, đơn vị và loại lương
    df_jobs['salary_min_mil'] = np.nan # Dùng np.nan cho kiểu float, đặt tên rõ hơn
    df_jobs['salary_max_mil'] = np.nan 
    df_jobs['salary_currency'] = None # Sẽ điền sau nếu phát hiện đơn vị khác
    df_jobs['salary_type'] = None 

    # Xem các giá trị duy nhất của cột salary để hiểu rõ hơn
    print("\nCác giá trị duy nhất trong cột 'salary' (trước khi xử lý):")
    unique_salaries = df_jobs['salary'].unique()
    print(unique_salaries[:min(20, len(unique_salaries))]) # In tối đa 20 giá trị đầu tiên

    # Hàm để trích xuất số từ chuỗi lương và xác định đơn vị
    def parse_salary_string(salary_text):
        if pd.isna(salary_text):
            return [], None, None # numbers, currency, original_type

        text_lower = str(salary_text).lower()
        
        numbers = []
        currency = "triệu VNĐ" # Mặc định
        original_type_guess = "Không xác định"

        # Ưu tiên xử lý "Thoả thuận" trước
        if "thoả thuận" in text_lower or "thỏa thuận" in text_lower:
            return [], currency, "Thoả thuận"

        # Kiểm tra đơn vị khác (ví dụ USD) - Phần này có thể mở rộng
        if "usd" in text_lower:
            currency = "USD"
            text_to_extract = text_lower.replace("usd", "").strip()
        elif "triệu" in text_lower: # Mặc định là triệu nếu không có đơn vị khác
            currency = "triệu VNĐ"
            text_to_extract = text_lower.replace("triệu", "").replace("vnd","").strip()
        else: # Nếu không có "triệu" hay "usd", giả định là đơn vị triệu VNĐ và text chỉ chứa số
            text_to_extract = text_lower.replace("vnd","").strip()


        # Loại bỏ các ký tự không phải số hoặc dấu chấm (cho số thập phân) hoặc dấu gạch ngang (cho khoảng)
        # Giữ lại dấu '-' để phân biệt khoảng
        # Regex này tìm các cụm số (có thể có dấu chấm thập phân)
        raw_numbers_str = re.findall(r'\d+\.?\d*', text_to_extract)
        numbers = [float(num_str) for num_str in raw_numbers_str]

        # Đoán loại lương dựa trên từ khóa và số lượng số
        if "tới" in text_lower or "upto" in text_lower or "đến" in text_lower:
            original_type_guess = "Tối đa"
        elif "từ" in text_lower and "đến" not in text_lower and "tới" not in text_lower : # Xử lý "Từ X"
             original_type_guess = "Tối thiểu"
        elif len(numbers) == 2:
            original_type_guess = "Khoảng"
        elif len(numbers) == 1 and not ("tới" in text_lower or "upto" in text_lower or "đến" in text_lower or "từ" in text_lower):
            original_type_guess = "Cố định"
        
        return numbers, currency, original_type_guess

    # Lặp qua DataFrame để xử lý từng dòng
    for index, row in df_jobs.iterrows():
        salary_str_original = row['salary'] # Giữ lại salary gốc để tham chiếu
        
        numbers, currency_detected, type_guess = parse_salary_string(salary_str_original)
        
        df_jobs.loc[index, 'salary_type'] = type_guess
        df_jobs.loc[index, 'salary_currency'] = currency_detected

        if type_guess == 'Thoả thuận':
            df_jobs.loc[index, 'salary_min_mil'] = np.nan
            df_jobs.loc[index, 'salary_max_mil'] = np.nan
        elif type_guess == 'Tối đa':
            if numbers:
                df_jobs.loc[index, 'salary_max_mil'] = numbers[0]
                df_jobs.loc[index, 'salary_min_mil'] = np.nan # Hoặc 0, hoặc một phần trăm của max
        elif type_guess == 'Tối thiểu':
            if numbers:
                df_jobs.loc[index, 'salary_min_mil'] = numbers[0]
                df_jobs.loc[index, 'salary_max_mil'] = np.nan # Max không xác định
        elif type_guess == 'Khoảng':
            if len(numbers) == 2:
                df_jobs.loc[index, 'salary_min_mil'] = min(numbers)
                df_jobs.loc[index, 'salary_max_mil'] = max(numbers)
            elif len(numbers) == 1: # Trường hợp hiếm: "X - Y triệu" nhưng chỉ parse được 1 số
                df_jobs.loc[index, 'salary_min_mil'] = numbers[0]
                df_jobs.loc[index, 'salary_max_mil'] = numbers[0] 
                df_jobs.loc[index, 'salary_type'] = 'Cố định/Lỗi parse khoảng'
        elif type_guess == 'Cố định':
            if numbers:
                df_jobs.loc[index, 'salary_min_mil'] = numbers[0]
                df_jobs.loc[index, 'salary_max_mil'] = numbers[0]
        else: # Không xác định hoặc lỗi parse
            # Giữ nguyên là np.nan nếu không parse được
            if numbers: # Nếu vẫn parse ra số nhưng không khớp type
                 print(f"  Lương có số nhưng không rõ loại: '{salary_str_original}' -> numbers: {numbers} tại index {index}")
            # else: # Bỏ print này nếu không muốn thấy quá nhiều log cho trường hợp không có số
            #     print(f"  Không thể xử lý salary: '{salary_str_original}' tại index {index}")
            pass


    print("\n--- DataFrame sau khi xử lý cột 'salary' ---")
    # Hiển thị các cột mới và 15 dòng đầu
    display(df_jobs[['salary', 'salary_min_mil', 'salary_max_mil', 'salary_currency', 'salary_type']].head(15))

    print("\nKiểm tra kiểu dữ liệu của các cột salary mới:")
    print(df_jobs[['salary_min_mil', 'salary_max_mil']].dtypes) # salary_min_mil và salary_max_mil nên là float64
    
    print("\nThống kê mô tả cho salary_min_mil và salary_max_mil:")
    display(df_jobs[['salary_min_mil', 'salary_max_mil']].describe())
    
    print("\nKiểm tra các giá trị 'salary_type' đã phân loại:")
    print(df_jobs['salary_type'].value_counts(dropna=False))

else:
    print("DataFrame 'df_jobs' không tồn tại hoặc rỗng. Hãy chạy lại Cell 6 (cũ) để đọc dữ liệu.")

--- Bắt đầu xử lý cột 'salary' ---

Các giá trị duy nhất trong cột 'salary' (trước khi xử lý):
['40 - 50 triệu' '50 - 200 triệu' 'Tới 80 triệu' 'Từ 50 triệu'
 'Thoả thuận' '10 - 50 triệu' '9 - 50 triệu' '10 - 13 triệu'
 '20 - 100 triệu' '10 - 15 triệu' '10 - 30 triệu' 'Từ 8.5 triệu'
 'Từ 20 triệu' '12 - 35 triệu' 'Tới 20 triệu' '25 - 80 triệu'
 'Từ 25 triệu' 'Tới 100 triệu' '5 - 15 triệu' '8 - 20 triệu']

--- DataFrame sau khi xử lý cột 'salary' ---


,salary,salary_min_mil,salary_max_mil,salary_currency,salary_type
0,40 - 50 triệu,40.0,50.0,triệu VNĐ,Khoảng
1,50 - 200 triệu,50.0,200.0,triệu VNĐ,Khoảng
2,Tới 80 triệu,NaN,80.0,triệu VNĐ,Tối đa
3,Từ 50 triệu,50.0,NaN,triệu VNĐ,Tối thiểu
4,Thoả thuận,NaN,NaN,triệu VNĐ,Thoả thuận
5,10 - 50 triệu,10.0,50.0,triệu VNĐ,Khoảng
6,9 - 50 triệu,9.0,50.0,triệu VNĐ,Khoảng
7,50 - 200 triệu,50.0,200.0,triệu VNĐ,Khoảng
8,10 - 13 triệu,10.0,13.0,triệu VNĐ,Khoảng
9,20 - 100 triệu,20.0,100.0,triệu VNĐ,Khoảng



Kiểm tra kiểu dữ liệu của các cột salary mới:
salary_min_mil    float64
salary_max_mil    float64
dtype: object

Thống kê mô tả cho salary_min_mil và salary_max_mil:


,salary_min_mil,salary_max_mil
count,40.000000,38.000000
mean,17.012500,46.500000
std,13.957577,46.710002
min,3.000000,4.000000
25%,8.500000,15.000000
50%,10.000000,30.000000
75%,20.000000,50.000000
max,50.000000,200.000000



Kiểm tra các giá trị 'salary_type' đã phân loại:
salary_type
Khoảng        34
Tối thiểu      6
Thoả thuận     6
Tối đa         4
Name: count, dtype: int64


# Cell 8: Detailed Processing for 'post_date' column

In [32]:
if 'df_jobs' in locals() and not df_jobs.empty:
    if 'scrape_date' not in df_jobs.columns:
        print("Lỗi: Cột 'scrape_date' không tồn tại trong DataFrame.")
        print("Hãy đảm bảo Cell 4 đã được cập nhật để thêm cột này và Cell 5, 6 đã chạy lại với dữ liệu mới.")
    else:
        print("--- Bắt đầu xử lý cột 'post_date' sử dụng 'scrape_date' ---")
        
        print("\nCác giá trị duy nhất trong cột 'post_date' (trước khi xử lý):")
        unique_post_dates = df_jobs['post_date'].unique()
        print(unique_post_dates[:min(20, len(unique_post_dates))])

        df_jobs['calculated_post_date'] = pd.NaT 
        print(f"\nSẽ sử dụng 'scrape_date' làm mốc tính toán.")

        for index, row in df_jobs.iterrows():
            date_str_from_scrape = row['post_date']
            scrape_date_str_from_df = row['scrape_date'] 
            
            calculated_date_val = pd.NaT 

            if pd.isna(date_str_from_scrape) or pd.isna(scrape_date_str_from_df):
                df_jobs.loc[index, 'calculated_post_date'] = pd.NaT
                continue

            try:
                scrape_datetime_object = datetime.strptime(scrape_date_str_from_df, '%Y-%m-%d')
            except ValueError:
                print(f"  Lỗi định dạng scrape_date: '{scrape_date_str_from_df}' tại index {index}. Bỏ qua tính toán cho job này.")
                df_jobs.loc[index, 'calculated_post_date'] = pd.NaT
                continue

            date_str_lower_case = str(date_str_from_scrape).lower()

            try:
                if "hôm qua" in date_str_lower_case:
                    calculated_date_val = scrape_datetime_object - timedelta(days=1)
                elif "hôm nay" in date_str_lower_case: 
                    calculated_date_val = scrape_datetime_object
                elif "ngày trước" in date_str_lower_case:
                    days_ago_match = re.search(r'(\d+)\s*ngày trước', date_str_lower_case)
                    if days_ago_match:
                        days = int(days_ago_match.group(1))
                        calculated_date_val = scrape_datetime_object - timedelta(days=days)
                elif "tuần trước" in date_str_lower_case:
                    weeks_ago_match = re.search(r'(\d+)\s*tuần trước', date_str_lower_case)
                    if weeks_ago_match:
                        weeks = int(weeks_ago_match.group(1))
                        calculated_date_val = scrape_datetime_object - timedelta(weeks=weeks)
                elif "tháng trước" in date_str_lower_case:
                    months_ago_match = re.search(r'(\d+)\s*tháng trước', date_str_lower_case)
                    if months_ago_match:
                        months = int(months_ago_match.group(1))
                        calculated_date_val = scrape_datetime_object - timedelta(days=int(months * 30.4375)) 
                else:
                    try:
                        parsed_directly = pd.to_datetime(date_str_from_scrape, dayfirst=True, errors='coerce')
                        if pd.notna(parsed_directly):
                             calculated_date_val = parsed_directly
                        else: 
                             parsed_directly_alt = pd.to_datetime(date_str_from_scrape, errors='coerce')
                             if pd.notna(parsed_directly_alt):
                                 calculated_date_val = parsed_directly_alt
                             # else: # Bỏ print ở đây để tránh log quá nhiều nếu nhiều trường hợp không parse được
                                 # print(f"  Không thể chuyển đổi post_date trực tiếp: '{date_str_from_scrape}' tại index {index}")
                    except ValueError:
                        # print(f"  Lỗi ValueError khi cố parse post_date trực tiếp: '{date_str_from_scrape}' tại index {index}")
                        pass # Bỏ qua nếu không parse được
                
                # Bỏ .normalize() 
                if pd.notna(calculated_date_val):
                    # Nếu calculated_date_val là datetime.datetime, gán trực tiếp
                    # Nếu nó là Pandas Timestamp (từ pd.to_datetime), thì .normalize() sẽ hoạt động
                    if isinstance(calculated_date_val, pd.Timestamp):
                        df_jobs.loc[index, 'calculated_post_date'] = calculated_date_val.normalize()
                    else: # Là datetime.datetime chuẩn
                        df_jobs.loc[index, 'calculated_post_date'] = calculated_date_val 

            except Exception as e_date_processing:
                print(f"  Lỗi không xác định khi xử lý post_date '{date_str_from_scrape}' tại index {index}: {e_date_processing}")
                df_jobs.loc[index, 'calculated_post_date'] = pd.NaT


        print("\n--- DataFrame sau khi xử lý cột 'post_date' ---")
        display(df_jobs[['post_date', 'scrape_date', 'calculated_post_date']].head(15))

        print("\nKiểm tra kiểu dữ liệu của cột 'calculated_post_date':")
        print(df_jobs['calculated_post_date'].dtype) 
        
        print("\nKiểm tra các giá trị bị thiếu trong 'calculated_post_date':")
        print(f"Số lượng NaT (ngày không xác định): {df_jobs['calculated_post_date'].isnull().sum()}")

        print("\nThống kê mô tả cho 'calculated_post_date' (nếu có giá trị hợp lệ):")
        if df_jobs['calculated_post_date'].notnull().any():
            # Bỏ datetime_is_numeric=True 
            display(df_jobs['calculated_post_date'].describe()) 
        else:
            print("Không có giá trị calculated_post_date hợp lệ để thống kê.")
else:
    print("DataFrame 'df_jobs' không tồn tại hoặc rỗng. Hãy chạy lại các cell trước để tạo và đọc df_jobs.")

--- Bắt đầu xử lý cột 'post_date' sử dụng 'scrape_date' ---

Các giá trị duy nhất trong cột 'post_date' (trước khi xử lý):
['5 ngày trước' '6 ngày trước' '1 tuần trước' '1 ngày trước'
 '4 ngày trước' '2 tuần trước' '2 ngày trước']

Sẽ sử dụng 'scrape_date' làm mốc tính toán.

--- DataFrame sau khi xử lý cột 'post_date' ---


,post_date,scrape_date,calculated_post_date
0,5 ngày trước,2025-05-21,2025-05-16
1,6 ngày trước,2025-05-21,2025-05-15
2,1 tuần trước,2025-05-21,2025-05-14
3,1 tuần trước,2025-05-21,2025-05-14
4,1 tuần trước,2025-05-21,2025-05-14
5,1 tuần trước,2025-05-21,2025-05-14
6,1 tuần trước,2025-05-21,2025-05-14
7,1 tuần trước,2025-05-21,2025-05-14
8,1 ngày trước,2025-05-21,2025-05-20
9,1 tuần trước,2025-05-21,2025-05-14



Kiểm tra kiểu dữ liệu của cột 'calculated_post_date':
datetime64[ns]

Kiểm tra các giá trị bị thiếu trong 'calculated_post_date':
Số lượng NaT (ngày không xác định): 0

Thống kê mô tả cho 'calculated_post_date' (nếu có giá trị hợp lệ):


count                     50
mean     2025-05-12 07:40:48
min      2025-05-07 00:00:00
25%      2025-05-07 00:00:00
50%      2025-05-14 00:00:00
75%      2025-05-15 00:00:00
max      2025-05-20 00:00:00
Name: calculated_post_date, dtype: object